# 05 — Infer against the KServe endpoint

Call the deployed `xray-detector` InferenceService.

During canary (Serverless + tag routing), you can target:
- default URL — traffic split (e.g. 90/10)
- `latest-` host/path tag — canary revision
- `prev-` host/path tag — previous good revision

YOLO ONNX responses may need client-side postprocessing for pretty boxes; this notebook focuses on proving the serving path.

In [ ]:
import base64
import json
import os
from pathlib import Path

import numpy as np
import requests
from PIL import Image

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "knife.jpg").exists() and (REPO_ROOT.parent / "knife.jpg").exists():
    REPO_ROOT = REPO_ROOT.parent

ENDPOINT = os.environ.get("INFERENCE_ENDPOINT")
if not ENDPOINT:
    raise SystemExit("Set INFERENCE_ENDPOINT to the KServe predictor URL")

HOST = os.environ.get("INFERENCE_HOST_HEADER")
MODEL_NAME = os.environ.get("INFERENCE_SERVICE_NAME", "xray-detector")
IMAGE_PATH = REPO_ROOT / os.environ.get("INFER_IMAGE", "knife.jpg")
IMGSZ = int(os.environ.get("DEMO_IMGSZ", "640"))

print("ENDPOINT:", ENDPOINT)
print("IMAGE:", IMAGE_PATH)

In [ ]:
def load_image_b64(path: Path, size: int = 640) -> str:
    img = Image.open(path).convert("RGB").resize((size, size))
    import io

    buf = io.BytesIO()
    img.save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def load_image_chw(path: Path, size: int = 640) -> list:
    img = Image.open(path).convert("RGB").resize((size, size))
    arr = np.asarray(img).astype("float32") / 255.0
    chw = np.transpose(arr, (2, 0, 1))
    return chw.reshape(1, 3, size, size).tolist()


headers = {"Content-Type": "application/json"}
if HOST:
    headers["Host"] = HOST
token = os.environ.get("INFERENCE_TOKEN")
if token:
    headers["Authorization"] = f"Bearer {token}"

# Open Inference Protocol v2 tensor payload (works with many KServe runtimes).
# If your runtime expects raw/bytes, switch to the bytes payload below.
payload = {
    "inputs": [
        {
            "name": "images",
            "shape": [1, 3, IMGSZ, IMGSZ],
            "datatype": "FP32",
            "data": load_image_chw(IMAGE_PATH, IMGSZ),
        }
    ]
}

url = ENDPOINT.rstrip("/")
if not url.endswith("/infer"):
    url = f"{url}/v2/models/{MODEL_NAME}/infer"

resp = requests.post(url, headers=headers, data=json.dumps(payload), timeout=120)
print("status:", resp.status_code)
print(resp.text[:2000] if resp.text else "(empty)")
resp.raise_for_status()

## Optional: tag-routed canary check

If `serving.kserve.io/enable-tag-routing=true`, set `INFERENCE_ENDPOINT_LATEST` / `INFERENCE_ENDPOINT_PREV` and re-run the request cell against each.

In [ ]:
for label, env_key in [("latest/canary", "INFERENCE_ENDPOINT_LATEST"), ("prev", "INFERENCE_ENDPOINT_PREV")]:
    ep = os.environ.get(env_key)
    if not ep:
        print(f"skip {label}: {env_key} not set")
        continue
    u = ep.rstrip("/")
    if not u.endswith("/infer"):
        u = f"{u}/v2/models/{MODEL_NAME}/infer"
    r = requests.post(u, headers=headers, data=json.dumps(payload), timeout=120)
    print(label, "status", r.status_code, "bytes", len(r.content))